#### Instead of loading full data load this instead
X = np.load("data/X_sample.npy")
Y = np.load("data/Y_sample.npy")

#### Libraries

In [1]:
import numpy as np
import pandas as pd
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import math
import random
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix

# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)


#### Data Import

In [2]:
import scipy.io as io
data = io.loadmat('../data/WLDataCW.mat') #import data
print('the length: ' + str(len(data)))# check the number of keys
print(data.keys()) # show the name of keys
print(data['__globals__'])
data_only = data['data'] # assign the data to a new variable
label = data['label'] # assign the label to a new variable
print(data_only.shape) # check the dimension of the data
print(label.shape) #check the dimension of the label


the length: 5
dict_keys(['__header__', '__version__', '__globals__', 'data', 'label'])
[]
(62, 512, 360)
(1, 360)


#### Feature Extraction

In [3]:
def extract_features(data_input):
    num_samples = data_input.shape[2]
    # Matrix X: rows are features (620), columns are samples (360)
    X = np.zeros((620, num_samples)) 
    
    for i in range(num_samples):
        trial_features = []
        for electrode in range(62):
            # Split 2s (512 points) into two 1s segments (256 points each)
            sec1 = data_input[electrode, :256, i]
            sec2 = data_input[electrode, 256:, i]
            
            for segment in [sec1, sec2]:
                # Apply FFT to get powers from 0Hz to 128Hz
                fft_vals = np.abs(np.fft.rfft(segment))
                
                # Average power for the 5 bands (Theta, Delta, Alpha, Beta, Gamma)
                # Mapping indices directly to Hz since segment length is 1s
                theta = np.mean(fft_vals[1:5])   # 0.5-4Hz
                delta = np.mean(fft_vals[4:9])   # 4-8Hz
                alpha = np.mean(fft_vals[8:14])  # 8-13Hz
                beta  = np.mean(fft_vals[13:31]) # 13-30Hz
                gamma = np.mean(fft_vals[30:])   # >30Hz
                
                trial_features.extend([theta, delta, alpha, beta, gamma])
        
        X[:, i] = trial_features
    return X

X_all = extract_features(data_only)
print(f"Feature Matrix X shape: {X_all.shape}")

Feature Matrix X shape: (620, 360)


#### Label Encoding

In [4]:
def one_hot_encode(labels_input):
    # Shape (2, 360) as per the instruction for matrix Y
    Y = np.zeros((2, labels_input.shape[1]))
    for i in range(labels_input.shape[1]):
        if labels_input[0, i] == 0:
            Y[:, i] = [1, 0]
        else:
            Y[:, i] = [0, 1]
    return Y

Y_all = one_hot_encode(label)
print(f"Ground-truth Matrix Y shape: {Y_all.shape}")

Ground-truth Matrix Y shape: (2, 360)


In [7]:
X_sample = X_all[:20]
Y_sample = Y_all[:20]

np.save("X_sample.npy", X_sample)
np.save("Y_sample.npy", Y_sample)

#### logistic Regression section

In [5]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward_propagation(X, W, b):
    Z = np.dot(W, X) + b
    A = sigmoid(Z)
    return A

def compute_cost(A, Y):
    m = Y.shape[1]
    # Small epsilon (1e-8) added to prevent log(0) errors
    cost = -(1/m) * np.sum(Y * np.log(A + 1e-8))
    return cost

def back_propagation(X, A, Y):
    m = Y.shape[1]
    dZ = A - Y
    dW = (1/m) * np.dot(dZ, X.T)
    db = (1/m) * np.sum(dZ, axis=1, keepdims=True)
    return dW, db

In [6]:
def run_classification_experiment(X, Y, epochs=2500, lr=0.01):
    fold_size = 72
    accuracies = []

    for fold in range(5):
        # 5-fold split (72 test / 288 train)
        start, end = fold * fold_size, (fold + 1) * fold_size
        X_test, Y_test = X[:, start:end], Y[:, start:end]
        X_train = np.delete(X, np.arange(start, end), axis=1)
        Y_train = np.delete(Y, np.arange(start, end), axis=1)

        # Initialize Weights (2 classes) and Bias
        W = np.zeros((2, X.shape[0]))
        b = np.zeros((2, 1))

        # Training Phase
        for epoch in range(epochs):
            A = forward_propagation(X_train, W, b)
            dW, db = back_propagation(X_train, A, Y_train)
            
            # GRADIENT DESCENT PARAMETER UPDATE
            W -= lr * dW
            b -= lr * db

        # Evaluation Phase
        A_test = forward_propagation(X_test, W, b)
        predictions = np.argmax(A_test, axis=0)
        actual_labels = np.argmax(Y_test, axis=0)
        accuracy = np.mean(predictions == actual_labels) * 100
        accuracies.append(accuracy)
        
        print(f"Fold {fold+1} Accuracy: {accuracy:.2f}%")

    print(f"\nFinal 5-Fold Average Accuracy: {np.mean(accuracies):.2f}%")

run_classification_experiment(X_all, Y_all)

C:\Users\adedo\AppData\Local\Temp\ipykernel_46712\2818421546.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


Fold 1 Accuracy: 98.61%
Fold 2 Accuracy: 81.94%
Fold 3 Accuracy: 80.56%
Fold 4 Accuracy: 83.33%
Fold 5 Accuracy: 98.61%

Final 5-Fold Average Accuracy: 88.61%


#### Neural network section

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

def build_mlp(input_dim):
    model = Sequential([
        # Input Layer and First Hidden Layer
        Dense(64, activation='relu', input_dim=input_dim),
        # Second Hidden Layer
        Dense(32, activation='relu'),
        # Output Layer (2 nodes for our [1,0] or [0,1] vectors)
        Dense(2, activation='softmax')
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [8]:
# Transpose X and Y for Keras compatibility (Samples, Features)
X_mlp = X_all.T 
Y_mlp = Y_all.T

fold_size = 72
mlp_accuracies = []

for fold in range(5):
    start, end = fold * fold_size, (fold + 1) * fold_size
    
    # Split Test and Train
    X_test_f, Y_test_f = X_mlp[start:end], Y_mlp[start:end]
    X_train_f = np.delete(X_mlp, np.arange(start, end), axis=0)
    Y_train_f = np.delete(Y_mlp, np.arange(start, end), axis=0)

    # Build and Train
    model = build_mlp(X_mlp.shape[1])
    # We use a small batch size because the dataset is small (360 samples)
    model.fit(X_train_f, Y_train_f, epochs=80, batch_size=16, verbose=0)

    # Evaluate
    _, acc = model.evaluate(X_test_f, Y_test_f, verbose=0)
    mlp_accuracies.append(acc * 100)
    print(f"MLP Fold {fold+1} Accuracy: {acc*100:.2f}%")

print(f"\nFinal MLP Average Accuracy: {np.mean(mlp_accuracies):.2f}%")

C:\Users\adedo\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


MLP Fold 1 Accuracy: 100.00%
MLP Fold 2 Accuracy: 88.89%
MLP Fold 3 Accuracy: 88.89%
MLP Fold 4 Accuracy: 93.06%
MLP Fold 5 Accuracy: 100.00%

Final MLP Average Accuracy: 94.17%
